# שלב 3 – תיוג עמדות מתוך הצעות חוק

אותה שיטה כמו במצעים (`03_label_platforms.ipynb`): לכל שאלה המודל מציג את 10 הצעות החוק הקרובות ביותר, ואתה מחליט מה העמדה.

**הבדלים מהמצעים:**
- כל הצעת חוק נכנסת פעם אחת למפלגה, גם אם חתמו עליה כמה ח"כים ממנה.
- לכל הצעה מוצגים הסטטוס (האם הפכה לחוק), האם ח"כ מהמפלגה הוא המגיש הראשי, ושמות החותמים מהמפלגה.
- הצעת חוק משקפת את הח"כ, לא בהכרח את המפלגה שבה הוא נמצא היום.

**התוצר:** `bill_coverage_<מפלגה>.csv`, עם מזהה ההצעה וקישור לאתר הכנסת.

In [ ]:
import os, re, sqlite3
import pandas as pd
from IPython.display import clear_output
from sentence_transformers import SentenceTransformer

pd.set_option('display.max_colwidth', None)
conn = sqlite3.connect("../data/election.db")
questions = pd.read_sql('SELECT id, axis, text FROM "QuizQuestion" ORDER BY id', conn)
model = SentenceTransformer("intfloat/multilingual-e5-base")
K = 10


def load_bills(party):
    """הצעות החוק של מפלגה, הצעה אחת לכל מזהה כנסת."""
    rows = pd.read_sql('''
        SELECT b.title, b.summary, b.status, b.isLeadSponsor, b.sourceUrl, b.knessetNum, b.year, c.name
        FROM "Bill" b JOIN "Candidate" c ON c.id = b.candidateId
        WHERE c.partyId = ?''', conn, params=(party,))
    rows['billId'] = rows['sourceUrl'].str.extract(r'lawitemid=(\d+)')[0]
    bills = rows.groupby('billId').agg(
        title=('title', 'first'),
        summary=('summary', lambda s: next((x for x in s if x), '')),
        status=('status', 'first'),
        lead=('isLeadSponsor', 'max'),            # מגיש ראשי אם לפחות אחד מהח"כים הוא הראשי
        sponsors=('name', lambda s: ', '.join(sorted(set(s)))),
        sourceUrl=('sourceUrl', 'first'),
        knessetNum=('knessetNum', 'first'),
        year=('year', 'first'),
    ).reset_index()
    return bills


def bill_text(b):
    """הטקסט שהמודל משווה לשאלה: הכותרת, ותחילת התקציר אם יש."""
    return b['title'] + (' — ' + b['summary'][:300] if b['summary'] else '')


def norm_title(title):
    """הכותרת בלי "הצעת" ובלי השנה, כדי לזהות את אותה הצעה שהוגשה שוב בכנסת אחרת."""
    s = re.sub(r",?\s*ה?תש[א-ת\"״׳']+\s*[-–]\s*\d{4}\s*$", "", title)
    s = re.sub(r"^הצעת\s+", "", s)
    return re.sub(r"[^\w\s]", "", s).strip()


def dedupe(bills):
    """מסיר הצעות שהוגשו שוב בכנסת אחרת באותה כותרת, ומחשב embeddings.

    לא לפי דמיון של המודל: כותרות של הצעות חוק בנויות מאותה תבנית
    ("הצעת חוק ... (תיקון ...), התשפ"ב-2022"), ולכן המודל רואה הצעות שונות
    כמעט זהות. בניסוי, סף 0.95 מחק 61% מההצעות של ש"ס.
    """
    bills = bills.assign(key=bills['title'].map(norm_title))
    bills = bills.sort_values('lead', ascending=False).drop_duplicates('key').sort_index()
    bills = bills.drop(columns='key').reset_index(drop=True)
    emb = model.encode(["passage: " + bill_text(b) for _, b in bills.iterrows()], normalize_embeddings=True)
    return bills, emb

## בחירת מפלגה
משנים רק את `PARTY`. התא מכין את ההצעות ומראה כמה כבר תויגו.

In [ ]:
PARTY = 'shas'
OUT = f'bill_coverage_{PARTY}.csv'

all_bills = load_bills(PARTY)
bills, passage_emb = dedupe(all_bills)
query_emb = model.encode(["query: " + t for t in questions['text']], normalize_embeddings=True)
sims = model.similarity(query_emb, passage_emb)
print(f"{PARTY}: {len(all_bills)} הצעות שונות, {len(bills)} אחרי הסרת הצעות שהוגשו שוב")
print(f"מגיש ראשי ב-{int(bills['lead'].sum())} מהן, {int((bills['status'] == 'התקבלה בקריאה שלישית').sum())} הפכו לחוק")

done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
print(f"כבר תויגו: {len(done)} מתוך {len(questions)}")

## התיוג

| מה רושמים | המשמעות |
|---|---|
| `1 23` | עמדה **בעד** ההיגד, לפי הצעה 23 |
| `-1 45` | עמדה **נגד** ההיגד, לפי הצעה 45 |
| `1 23 p` | בעד, אבל ההצעה תומכת רק **בחלק** מההיגד |
| `0 12` | ההצעה עוסקת בהיגד, אבל **הכיוון לא ברור** |
| Enter | אף הצעה לא עוסקת בהיגד |

קודם כיוון, אחר כך מספר הצעה, מופרדים ברווח.

In [ ]:
done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
rows = done.to_dict('records')
answered = {r['question_id'] for r in rows}

for row, q in questions.iterrows():
    if q['id'] in answered:
        continue
    clear_output(wait=True)
    print(f"[{len(rows) + 1}/{len(questions)}]  {q['id']} ({q['axis']}): {q['text']}\n")
    scores, idx = sims[row].topk(K)
    for rank, i in enumerate(idx.tolist(), start=1):
        b = bills.iloc[i]
        role = 'מגיש ראשי' if b['lead'] else 'חתום'
        print(f"{rank:>2}. [{i:>3}] {b['title']}")
        if b['summary']:
            print(f"        תקציר: {b['summary'][:250]}")
        print(f"        {b['status']} · {role} · {b['sponsors']}\n")

    while True:
        answer = input("עמדה ומספר הצעה (למשל: 1 23 | -1 45 p | 0 12 | Enter = לא מכוסה): ").split()
        if not answer:
            confirm = input(f"{q['id']} לא מכוסה? Enter לאישור, כל דבר אחר כדי לחזור: ")
            if not confirm:
                break
            continue
        if (len(answer) >= 2 and answer[0] in ('1', '-1', '0')
                and answer[1].isdigit() and int(answer[1]) < len(bills)):
            break
        print("לא תקין: קודם כיוון (1, -1 או 0), ואז מספר הצעה.")

    base = {'party': PARTY, 'question_id': q['id'], 'axis': q['axis']}
    if not answer:
        rows.append({**base, 'stance': None, 'confidence': 'not_covered'})
    else:
        stance, i = int(answer[0]), int(answer[1])
        b = bills.iloc[i]
        confidence = 'partial' if 'p' in answer[2:] else ('unclear' if stance == 0 else 'sure')
        rows.append({**base, 'stance': stance, 'confidence': confidence, 'bill': i,
                     'billId': b['billId'], 'title': b['title'], 'status': b['status'],
                     'lead': bool(b['lead']), 'sponsors': b['sponsors'], 'sourceUrl': b['sourceUrl']})
    pd.DataFrame(rows).to_csv(OUT, index=False)   # שמירה אחרי כל תשובה

clear_output()
print(f"סיימת! {len(rows)} שאלות נשמרו ב-{OUT}")

## תיקונים
אחרי הבדיקה. `fix('q08')` = לא מכוסה, `fix('q05', 1, 35, partial=True)` = בעד, חלקי, הצעה 35.

In [ ]:
def fix(qid, stance=None, i=None, partial=False):
    d = pd.read_csv(OUT)
    m = d['question_id'] == qid
    cols = ['stance', 'bill', 'billId', 'title', 'status', 'lead', 'sponsors', 'sourceUrl']
    for col in cols:
        if col not in d.columns:
            d[col] = None
    d[cols] = d[cols].astype(object)
    if stance is None:
        d.loc[m, cols] = None
        d.loc[m, 'confidence'] = 'not_covered'
    else:
        b = bills.iloc[i]
        d.loc[m, cols] = [stance, i, b['billId'], b['title'], b['status'], bool(b['lead']), b['sponsors'], b['sourceUrl']]
        d.loc[m, 'confidence'] = 'partial' if partial else ('unclear' if stance == 0 else 'sure')
    d.to_csv(OUT, index=False)

In [ ]:
d = pd.read_csv(OUT)
print(d['confidence'].value_counts().to_string())
d[d['confidence'] != 'not_covered'][['question_id', 'stance', 'confidence', 'title', 'lead']]